<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Ein Regressionsmodell trainieren

In dieser Learning Experience wird die Objekterkennung als vorgegebene Blackbox behandelt. Du musst den Detektor selbst **nicht** trainieren, ersetzen oder verbessern.

Deine Aufgabe ist es, einen **Distanz-Regressor** zu bauen, der nur einfache Detektionsausgaben als Eingaben verwendet:

- `pixel_width`
- `pixel_height`

Auf Basis dieser beiden Merkmale sollst du ein Regressionsmodell trainieren, das die Distanz zum erkannten Objekt vorhersagt. Das Modell soll mit **gradient descent** trainiert werden, und die gelernten Parameter sollen anschliessend mit **NumPy** exportiert werden, damit sie spaeter vom Laufzeitcode geladen werden koennen.

Als Kontext fuer diese Features gilt: Ein Bild ist in dieser Aufgabe maximal `224 x 224` Pixel gross. Sinnvolle Werte fuer `pixel_width` und `pixel_height` liegen also ebenfalls in diesem Bereich.


## Lernziele

Nach der Bearbeitung dieses Notebooks solltest du in der Lage sein:

- zu erklaeren, warum diese Aufgabe ein Regressionsproblem und kein Objekterkennungsproblem ist
- Bounding-Box-Breite und -Hoehe als Eingabemerkmale fuer ein einfaches Modell zu verwenden
- ein kleines Regressionsmodell mit gradient descent zu trainieren
- gelernte Gewichte mit `numpy.savez(...)` zu exportieren
- die Modellparameter fuer den spaeteren Import mit `numpy.load(...)` im Integrationscode vorzubereiten


#### Datensatz bereitstellen

Wir werden dein Regressionsmodell direkt in dieser Notebook-Umgebung trainieren.

Fuer diese angepasste Version der Learning Experience kannst du direkt mit einer vorbereiteten synthetischen CSV-Datei arbeiten. Sie liegt im Ordner `assets/data` und enthaelt 1000 kuenstlich generierte Beispiele mit `pixel_width`, `pixel_height` und `distance_m`.

Du kannst diese CSV-Datei direkt mit NumPy laden und sofort mit der Visualisierung und dem Training beginnen.


## Daten vor dem Training visualisieren

Bevor du das Modell trainierst, solltest du dir die Datenpunkte anschauen. So bekommst du ein Gefuehl dafuer, wie `pixel_width`, `pixel_height` und `distance_m` zusammenhaengen.

Die folgenden Zellen laden die synthetische CSV-Datei und visualisieren die Daten mit Matplotlib.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = np.genfromtxt(
    "../../assets/data/synthetic_distance_data.csv",
    delimiter=";",
    names=True
)

pixel_width = data["pixel_width"]
pixel_height = data["pixel_height"]
distance_m = data["distance_m"]

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    pixel_width,
    pixel_height,
    c=distance_m,
    cmap="viridis",
    alpha=0.75,
    edgecolors="none"
)
plt.colorbar(scatter, label="Distance [m]")
plt.xlabel("Bounding box width [px]")
plt.ylabel("Bounding box height [px]")
plt.title("Synthetic training data before regression")
plt.xlim(0, 224)
plt.ylim(0, 224)
plt.grid(alpha=0.2)
plt.show()


Die naechsten beiden Plots zeigen die Distanz einmal in Abhaengigkeit von der Bounding-Box-Breite und einmal in Abhaengigkeit von der Bounding-Box-Hoehe. So kannst du direkt sehen, wie stark beide Merkmale mit der Zielgroesse zusammenhaengen.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(pixel_width, distance_m, alpha=0.65, color="tab:blue", edgecolors="none")
axes[0].set_title("Distance as a function of bounding box width")
axes[0].set_xlabel("Bounding box width [px]")
axes[0].set_ylabel("Distance [m]")
axes[0].set_xlim(0, 224)
axes[0].grid(alpha=0.2)

axes[1].scatter(pixel_height, distance_m, alpha=0.65, color="tab:orange", edgecolors="none")
axes[1].set_title("Distance as a function of bounding box height")
axes[1].set_xlabel("Bounding box height [px]")
axes[1].set_ylabel("Distance [m]")
axes[1].set_xlim(0, 224)
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()


## Erforderliches Exportformat fuer die gelernten Gewichte

Sobald dein Training mit gradient descent konvergiert ist, exportiere die gelernten Parameter mit NumPy. Das ist ein verpflichtender Teil der Aufgabe, weil der Laufzeitcode diese Parameter spaeter direkt laedt.

Ein empfohlenes Exportformat ist:

```python
import numpy as np

weights = np.array([w_width, w_height], dtype=np.float32)
bias = np.array([b], dtype=np.float32)

np.savez("numpy_weights.npz", weights=weights, bias=bias)
```

Hinweise:

- `weights` sollte die gelernten Koeffizienten fuer Breite und Hoehe enthalten
- `bias` sollte den gelernten Intercept-Term enthalten
- verwende stabile Key-Namen, weil der Integrationscode diese spaeter ueber ihren Namen laedt
- die erzeugte Datei sollte spaeter dort abgelegt werden, wo `DISTANCE_MODEL_PATH` sie erwartet


## Wie der spaetere Import funktionieren soll

In `packages/solution/student_distance_estimator.py` sollen die Gewichte spaeter ebenfalls mit NumPy geladen werden. Ein typisches Muster ist:

```python
params = np.load(DISTANCE_MODEL_PATH)
weights = params["weights"]
bias = params["bias"]
```

Entscheidend ist die Konsistenz zwischen Export und Import:

- wenn du `weights=...` speicherst, musst du spaeter `params["weights"]` laden
- wenn du `bias=...` speicherst, musst du spaeter `params["bias"]` laden
- wenn du andere Namen waehlst, musst du den Laufzeitcode entsprechend anpassen

Ein einfaches Dateiformat macht es viel leichter, Notebook-Training und Laufzeit-Inferenz sauber miteinander zu verbinden.


#### Optional (nicht offiziell unterstuetzt) - lokales Training

Das ist nur fuer erfahrene Studierende empfohlen.
Weil diese Aufgabe nur ein kleines Regressionsmodell erfordert, ist lokales Training deutlich einfacher als ein komplettes Detector-Training-Setup.

Wenn du lokal trainierst, achte trotzdem darauf, dass du alle Anforderungen der Aufgabe erfuellst:

- trainiere das Distanzmodell mit gradient descent
- verwende nur Breite und Hoehe als Features
- exportiere die gelernten Parameter mit NumPy
- halte die exportierten Keys kompatibel mit dem spaeteren Import-Code


# Debugging und Modellinspektion

Sobald du mit dem Training fertig bist, gibt es ein paar Dinge, die du dir genau anschauen solltest.

* Pruefe, ob der Loss waehrend gradient descent sinkt.
* Schaue nach, ob die gelernten Gewichte plausible Vorzeichen und Groessenordnungen haben.
* Verifiziere, dass deine exportierte `.npz`-Datei wirklich die erwarteten Keys enthaelt.

Ein sehr nuetzlicher Debugging-Schritt ist es, die exportierte Datei direkt zu oeffnen und zu inspizieren:

```python
params = np.load("numpy_weights.npz")
print(params.files)
print(params["weights"])
print(params["bias"])
```

So kannst du Formatfehler erkennen, bevor du mit dem Integrationsschritt weitermachst.

Du kannst ausserdem deine Trainingsdaten oder Vorhersagen visualisieren, um zu pruefen, ob Breite und Hoehe fuer dein Setup informative Features sind.


## Bevor du weitermachst

Bevor du mit der Integration weitermachst, sollte alles Folgende erfuellt sein:

- du hast ein Regressionsmodell trainiert, keinen Detektor
- dein Modell verwendet nur Bounding-Box-Breite und -Hoehe als Eingaben
- die Parameter wurden mit gradient descent gelernt
- die finalen Parameter wurden mit NumPy exportiert
- du kennst die exakten Keys, die in der `.npz`-Datei verwendet wurden


# Naechster Schritt

Weiter zum [Integration notebook](../02-Integration/integration.ipynb)!